# 05 — Supervised data preparation

This notebook:
- loads the SBS96 matrix and merged clinical data,
- defines the binary smoking target,
- creates train, validation, and test splits at patient level,
- prepares the feature matrices,
- saves the split tables for the model notebook.


In [ ]:
%matplotlib inline

from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

def show_image(path, width=1000):
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Image not found: {path}")

def show_images(*paths, width=1000):
    for path in paths:
        show_image(path, width=width)

def get_sbs_columns(columns):
    cols = [col for col in columns if str(col).startswith("SBS") and str(col)[3:].isdigit()]
    return sorted(cols, key=lambda col: int(str(col)[3:]))

def normalize_rows(frame):
    frame = frame.copy()
    frame = frame.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    row_sums = frame.sum(axis=1)
    frame = frame.loc[row_sums > 0].copy()
    return frame.div(frame.sum(axis=1), axis=0)

def extract_patient_id(value):
    match = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", str(value))
    return match.group(1) if match else np.nan

def map_smoking_label(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().lower()

    if text in {"", "nan", "not reported", "unknown"}:
        return "Unknown"
    if "never" in text or "lifelong" in text or "non-smoker" in text or "nonsmoker" in text:
        return "Never"
    if "current reformed" in text or "reformed" in text or "former" in text:
        return "Former"
    if "current" in text:
        return "Current"
    return "Unknown"

def set_spaced_xticks(ax, labels, step=3):
    ax.set_xticks(np.arange(len(labels)) + 0.5)
    ax.set_xticklabels(labels, rotation=90)
    for i, label in enumerate(ax.get_xticklabels()):
        if i % step != 0:
            label.set_visible(False)

    from sklearn.model_selection import train_test_split


## 1. Define paths


In [ ]:
matrix_candidates = [
    PROJECT_ROOT / "data" / "maf" / "output" / "SBS" / "LUAD.SBS96.all",
    PROJECT_ROOT / "data" / "maf" / "output" / "SBS" / "TCGA_LUAD.SBS96.all",
]

sbs96_path = next((path for path in matrix_candidates if path.exists()), None)
clinical_path = PROJECT_ROOT / "data" / "clinical_exposure_merged.tsv"
split_output_dir = PROJECT_ROOT / "results" / "brf_split_binary"

split_output_dir.mkdir(parents=True, exist_ok=True)

print("SBS96 file   :", sbs96_path)
print("Clinical file:", clinical_path)
print("Split output :", split_output_dir)


## 2. Load the input tables


In [ ]:
if sbs96_path is None:
    raise FileNotFoundError("The SBS96 matrix file was not found.")

sbs = pd.read_csv(sbs96_path, sep="\t", index_col=0)
clinical = pd.read_csv(clinical_path, sep="\t")

if sbs.shape[1] != 96 and sbs.shape[0] == 96:
    sbs = sbs.T

if sbs.shape[1] != 96:
    raise RuntimeError(f"Expected 96 SBS columns, got shape {sbs.shape}")

sbs_cols = list(sbs.columns)

print("SBS matrix shape:", sbs.shape)
print("Clinical shape  :", clinical.shape)
print()

display(sbs.iloc[:5, :8])
display(clinical.head())


## 3. Merge SBS data with clinical data


In [ ]:
sbs = sbs.copy()
sbs["Patient_ID"] = sbs.index.astype(str).str[:12]

clinical = clinical.copy()
clinical["Patient_ID"] = clinical["cases.submitter_id"].astype(str).str[:12]

merged = pd.merge(sbs, clinical, on="Patient_ID", how="inner")

print("Merged shape:", merged.shape)
display(merged[["Patient_ID", "demographic.age_at_index", "demographic.gender", "exposures.tobacco_smoking_status"]].head())


## 4. Define the binary target


In [ ]:
merged["Smoking_3"] = merged["exposures.tobacco_smoking_status"].map(map_smoking_label)
merged = merged[merged["Smoking_3"].isin(["Never", "Former", "Current"])].copy()
merged["Smoking_Bin"] = (merged["Smoking_3"] != "Never").astype(int)

class_summary = (
    merged["Smoking_Bin"]
    .value_counts()
    .rename(index={0: "Never", 1: "Ever"})
    .rename_axis("class")
    .reset_index(name="n_rows")
)

print("Rows after filtering unknown smoking status:", len(merged))
display(class_summary)

display(
    merged["Smoking_3"]
    .value_counts()
    .rename_axis("Smoking_3")
    .reset_index(name="n_rows")
)


## 5. Create patient-level train, validation, and test splits


In [ ]:
RANDOM_STATE = 42

patient_labels = merged[["Patient_ID", "Smoking_Bin"]].drop_duplicates("Patient_ID")

trainval_patients, test_patients = train_test_split(
    patient_labels,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=patient_labels["Smoking_Bin"],
)

train_patients, val_patients = train_test_split(
    trainval_patients,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=trainval_patients["Smoking_Bin"],
)

train_ids = set(train_patients["Patient_ID"])
val_ids = set(val_patients["Patient_ID"])
test_ids = set(test_patients["Patient_ID"])

print("Leakage checks")
print("train ∩ val :", len(train_ids & val_ids))
print("train ∩ test:", len(train_ids & test_ids))
print("val   ∩ test:", len(val_ids & test_ids))


## 6. Build row-level split tables and save them


In [ ]:
train_df = merged[merged["Patient_ID"].isin(train_ids)].copy()
val_df = merged[merged["Patient_ID"].isin(val_ids)].copy()
test_df = merged[merged["Patient_ID"].isin(test_ids)].copy()

train_df.to_csv(split_output_dir / "train.csv", index=False)
val_df.to_csv(split_output_dir / "val.csv", index=False)
test_df.to_csv(split_output_dir / "test.csv", index=False)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "n_rows": len(train_df),
            "n_patients": train_df["Patient_ID"].nunique(),
            "n_never": int((train_df["Smoking_Bin"] == 0).sum()),
            "n_ever": int((train_df["Smoking_Bin"] == 1).sum()),
        },
        {
            "split": "validation",
            "n_rows": len(val_df),
            "n_patients": val_df["Patient_ID"].nunique(),
            "n_never": int((val_df["Smoking_Bin"] == 0).sum()),
            "n_ever": int((val_df["Smoking_Bin"] == 1).sum()),
        },
        {
            "split": "test",
            "n_rows": len(test_df),
            "n_patients": test_df["Patient_ID"].nunique(),
            "n_never": int((test_df["Smoking_Bin"] == 0).sum()),
            "n_ever": int((test_df["Smoking_Bin"] == 1).sum()),
        },
    ]
)

split_summary.to_csv(split_output_dir / "split_summary.tsv", sep="\t", index=False)
display(split_summary)


## 7. Prepare the feature matrices


In [ ]:
def prep_features(frame, sbs_cols, age_median=None, gender_fill_value=None):
    frame = frame.copy()

    frame["demographic.age_at_index"] = pd.to_numeric(
        frame["demographic.age_at_index"],
        errors="coerce",
    )

    frame["demographic.gender"] = (
        frame["demographic.gender"]
        .astype(str)
        .str.lower()
        .map({"male": 1, "female": 0})
    )

    if age_median is None:
        age_median = frame["demographic.age_at_index"].median()

    if gender_fill_value is None:
        mode_values = frame["demographic.gender"].mode()
        gender_fill_value = mode_values.iloc[0] if not mode_values.empty else 0

    frame["demographic.age_at_index"] = frame["demographic.age_at_index"].fillna(age_median)
    frame["demographic.gender"] = frame["demographic.gender"].fillna(gender_fill_value)

    x_sbs = frame[sbs_cols].copy()
    x_sbs = x_sbs.div(x_sbs.sum(axis=1), axis=0).fillna(0.0)

    x_clin = pd.DataFrame(
        {
            "age_years": frame["demographic.age_at_index"],
            "demographic.gender": frame["demographic.gender"],
        },
        index=frame.index,
    )

    x = pd.concat([x_sbs, x_clin], axis=1)
    y = frame["Smoking_Bin"].astype(int).to_numpy()

    return x, y, age_median, gender_fill_value


In [ ]:
X_train, y_train, age_median, gender_fill_value = prep_features(train_df, sbs_cols)
X_val, y_val, _, _ = prep_features(val_df, sbs_cols, age_median, gender_fill_value)
X_test, y_test, _, _ = prep_features(test_df, sbs_cols, age_median, gender_fill_value)

print("Feature matrix shapes")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape, "y_val  :", y_val.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)
print()
print("Imputation values")
print("age median       :", age_median)
print("gender fill value:", gender_fill_value)

display(X_train.iloc[:5, :8])
